
# 🔍 XGBoost Model Interpretability with SHAP: Line-by-Line Explanation

This notebook explains how to interpret a trained XGBoost model using SHAP (SHapley Additive exPlanations), a popular tool for **explainable AI**.

---

### 📦 Installation and Imports

```python
!pip install shap xgboost --quiet
```
* Installs **SHAP** (for model explanation) and **XGBoost** (gradient boosting model) quietly without verbose output.

```python
import shap
import xgboost as xgb
import pandas as pd
import matplotlib.pyplot as plt
```
* Imports necessary packages:
  - `shap`: used for generating local and global explanations of model predictions.
  - `xgboost`: for loading the trained model.
  - `pandas`: for loading the dataset.
  - `matplotlib.pyplot`: for any additional plotting.

---

### 📥 Load Dataset and Trained Model

```python
df = pd.read_csv('features.csv')
```
* Loads the dataset with engineered features.

```python
X = df.drop(columns=['Global_active_power', 'datetime', 'Unnamed: 0'])
```
* Removes the target variable and timestamp, leaving only the input features.

```python
model = xgb.Booster()
model.load_model('xgb_energy_model.json')
```
* Loads the pre-trained XGBoost model saved in JSON format.

---

# ✅ Align X with what the model expects
```python
X = X.loc[:, ~X.columns.duplicated()]  # Drop duplicate columns (if any)

# Ensure X is in the same order as model feature names
X = X[X.columns]  # Or use model.get_dump()[0] if feature_names is None
```
---
# Fallback: infer feature names from training data
# Save the original feature names during training if possible!
```python
model.feature_names = list(X.columns)  # Only safe if you're 100% sure X matches training layout
X.columns
```

### 🧠 SHAP Explainer and Value Computation

```python
#🧠 SHAP Explainer and Value Computation¶
explainer = shap.TreeExplainer(model)
#Creates a TreeExplainer, which is optimized for tree-based models like XGBoost.
shap_values = explainer.shap_values(X)
#Computes the SHAP values for each feature in each row.
#Each value explains how much a feature contributed to a prediction relative to the mean.
```
---

### 🐝 SHAP Beeswarm Plot

```python
shap.plots.beeswarm(shap.Explanation(values=shap_values, data=X, feature_names=X.columns))
```
* Shows a **summary plot** of feature impact:
  - Each dot is a SHAP value for a feature in one sample.
  - Color shows the feature value (e.g., high or low).
  - Spread shows how much features impact predictions.

---

### 📊 SHAP Bar Plot

```python
shap.plots.bar(shap.Explanation(values=shap_values, data=X, feature_names=X.columns))
```
* Plots **mean absolute SHAP values** of features.
* Ranks features by their average impact on the model’s predictions.

---

### 🔵 SHAP Scatter Plot

```python
# Get the index of the feature
feature_name = "hour_cos"
feature_index = X.columns.get_loc(feature_name)

# Build the full Explanation object
expl = shap.Explanation(values=shap_values,data=X.values,feature_names=X.columns)

# Extract Explanation slice safely (not Series or ndarray)
shap.plots.scatter(expl[:, feature_index])
```
* Visualizes how SHAP values for the feature `'hour_cos'` vary across data points.
* Helps understand how this time-based feature influences predictions.

---

#### 📈 SHAP Dependence Plot

```python
# This plot shows how the SHAP values for a given feature ('hour_cos') vary with the feature's actual values.
# The x-axis shows the raw feature value (e.g., cosine-transformed hour of day).
# The y-axis shows the SHAP value, which represents the feature's impact on the model's prediction.
# The color of each point optionally represents the value of an interacting feature (auto-detected by SHAP if not specified).
# This is helpful for:
# - Identifying non-linear effects
# - Detecting sharp thresholds or tipping points in feature influence
# - Visualizing interaction effects across two features

# 🔹 One-liner dependence plot
shap.dependence_plot("hour_cos", shap_values, X, feature_names=X.columns)
```
---


### 🌊 SHAP Waterfall Plot

```python
sample = X.iloc[[0]]  # Keep this as a 1-row DataFrame
explanation = explainer(sample)  # SHAP Explanation object with shape (1, n_features)

# Use index [0] to extract the first (and only) explanation from the matrix
shap.plots.waterfall(explanation[0])
```
* Shows a **step-by-step breakdown** of how each feature in a single sample contributes to the final prediction.
* Great for explaining individual predictions to stakeholders.

---


### 🌊 SHAP Scattered Plot - using matplotlib libraries

```python
#traditionall method usng matplotlib libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Data
feature_name = 'hour_cos'
feature_vals = X[feature_name].values
shap_vals = shap_values[:, X.columns.get_loc(feature_name)]

# Plot
plt.figure(figsize=(10, 6))
sns.scatterplot(
    x=feature_vals, 
    y=shap_vals, 
    s=30, 
    alpha=0.7, 
    color="#0288D1", 
    edgecolor="white", 
    linewidth=0.5
)

# Decorations
plt.title(f"SHAP Impact vs. Feature Value: '{feature_name}'", fontsize=16, fontweight='bold')
plt.xlabel(f"{feature_name} (normalized)", fontsize=14)
plt.ylabel("SHAP Value (Impact on Output)", fontsize=14)
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

---

### ✅ Summary

This script demonstrates:
1. Loading a trained model and dataset.
2. Using SHAP to compute feature contributions.
3. Visualizing local and global interpretability with beeswarm, bar, scatter, and waterfall plots.
